In [39]:
import numpy as np
import pandas as pd

## Round-Robin Miner Assignment

In [40]:
csv_path = '../data/bitcoin_blocks.csv'

df = pd.read_csv(csv_path)
df.head()

,block_id,n_transactions,size,difficulty,transaction_volume
0,0,619,1292812,1.000000e+00,5.139540e+03
1,1,5368,974804,1.233186e+07,1.235600e+06
2,2,4540,1916683,2.466371e+07,1.699461e+06
3,3,3044,1433631,3.699557e+07,1.783661e+06
4,4,3003,479187,4.932742e+07,1.601429e+06


In [41]:
# def preprocess(csv_path, n_miners=100):
#     df = pd.read_csv(csv_path)
#     df['miner_id'] = df['block_id'] % n_miners
#     return df

In [42]:
# Round-robin miner assignment for each block
df['miner_id'] = df['block_id'] % 100
df.head(10)

,block_id,n_transactions,size,difficulty,transaction_volume,miner_id
0,0,619,1292812,1.000000e+00,5.139540e+03,0
1,1,5368,974804,1.233186e+07,1.235600e+06,1
2,2,4540,1916683,2.466371e+07,1.699461e+06,2
3,3,3044,1433631,3.699557e+07,1.783661e+06,3
4,4,3003,479187,4.932742e+07,1.601429e+06,4
5,5,5955,294607,6.165928e+07,1.250893e+05,5
6,6,597,204560,7.399113e+07,6.052177e+05,6
7,7,4837,1717723,8.632299e+07,2.574254e+05,7
8,8,1398,470837,9.865484e+07,8.240118e+05,8
9,9,654,1513868,1.109867e+08,1.184718e+06,9


In [43]:
# loops back to miner_id 0 at block_id 100
df.iloc[97:105][['block_id', 'miner_id']]

,block_id,miner_id
97,97,97
98,98,98
99,99,99
100,100,0
101,101,1
102,102,2
103,103,3
104,104,4


## Aggregation

In [44]:
# usually transaction fees are a percentage of each transaction
# for simplicity, we just use transactions * transaction volume without multiplying a percentage since the ratios would be the same
df['fee_proxy'] = df['n_transactions'] * df['transaction_volume']
df

,block_id,n_transactions,size,difficulty,transaction_volume,miner_id,fee_proxy
0,0,619,1292812,1.000000e+00,5.139540e+03,0,3.181375e+06
1,1,5368,974804,1.233186e+07,1.235600e+06,1,6.632700e+09
2,2,4540,1916683,2.466371e+07,1.699461e+06,2,7.715552e+09
3,3,3044,1433631,3.699557e+07,1.783661e+06,3,5.429464e+09
4,4,3003,479187,4.932742e+07,1.601429e+06,4,4.809091e+09
...,...,...,...,...,...,...,...
810904,810904,2393,1532604,9.999951e+12,7.826699e+05,4,1.872929e+09
810905,810905,4158,649434,9.999963e+12,7.708996e+05,5,3.205401e+09
810906,810906,6181,11369,9.999975e+12,6.373446e+05,6,3.939427e+09
810907,810907,907,751999,9.999988e+12,6.624588e+05,7,6.008501e+08


In [45]:
minerdf_test = df.groupby('miner_id')
# groupby doesnt create a dataframe yet, it just tells pandas we're grouping by this
# 'miner_id' but we havent told it what to put in the columns and how to calculate
# the new dataframe data from the original dataframe.
type(minerdf_test)

pandas.api.typing.DataFrameGroupBy

In [46]:
miner_df = df.groupby('miner_id').agg(
    blocks_mined=('block_id', 'count'),
    avg_transactions=('n_transactions', 'mean'),
    avg_volume=('transaction_volume', 'mean'),
    avg_fee=('fee_proxy', 'mean'),
    fee_volatility=('fee_proxy', 'std'),
    avg_block_size=('size', 'mean'),
    difficulty=('difficulty', 'mean'),
    profitability=('fee_proxy', 'sum'),
    last_block_id=('block_id', 'max')   # last block a mined by a miner
).reset_index()

# convert profitability from total sum to avg profitability
miner_df['profitability'] = miner_df['profitability'] / (miner_df['blocks_mined'] + 1)
miner_df['age'] = df['block_id'].max() - miner_df['last_block_id']
miner_df.head(20)
miner_df['age'].describe()

count    100.000000
mean      49.500000
std       29.011492
min        0.000000
25%       24.750000
50%       49.500000
75%       74.250000
max       99.000000
Name: age, dtype: float64

## Calculating Efficiency Score and Binary Label

In [47]:
# added very small number (1 * 10^-9) to prevent division by zero
# if any miner has avg_volume of 0
miner_df['efficiency'] = miner_df['avg_transactions'] / (miner_df['avg_volume'] + 1e-9)
miner_df.sort_values(by='efficiency', ascending=False) # show by descending efficiency

,miner_id,blocks_mined,avg_transactions,avg_volume,avg_fee,fee_volatility,avg_block_size,difficulty,profitability,last_block_id,age,efficiency
28,28,8109,3482.256752,1.033372e+06,3.596305e+09,3.170148e+09,1.002843e+06,4.999679e+12,3.595862e+09,810828,80,0.003370
29,29,8109,3520.568011,1.047628e+06,3.688860e+09,3.219243e+09,1.000667e+06,4.999692e+12,3.688405e+09,810829,79,0.003361
67,67,8109,3472.035516,1.033257e+06,3.604440e+09,3.182318e+09,1.009763e+06,5.000160e+12,3.603996e+09,810867,41,0.003360
6,6,8110,3497.657707,1.041049e+06,3.651006e+09,3.171610e+09,1.007461e+06,5.000025e+12,3.650556e+09,810906,2,0.003360
77,77,8109,3480.787273,1.036851e+06,3.603228e+09,3.193937e+09,1.001647e+06,5.000284e+12,3.602784e+09,810877,31,0.003357
...,...,...,...,...,...,...,...,...,...,...,...,...
2,2,8110,3437.041060,1.056719e+06,3.633002e+09,3.187869e+09,1.012849e+06,4.999975e+12,3.632554e+09,810902,6,0.003253
45,45,8109,3427.352695,1.054067e+06,3.598595e+09,3.192410e+09,9.989339e+05,4.999889e+12,3.598152e+09,810845,63,0.003252
18,18,8109,3430.300530,1.056533e+06,3.634489e+09,3.216703e+09,1.001063e+06,4.999556e+12,3.634041e+09,810818,90,0.003247
68,68,8109,3438.549760,1.059645e+06,3.623344e+09,3.203651e+09,9.965575e+05,5.000173e+12,3.622897e+09,810868,40,0.003245


In [48]:
# Calculate median efficiency
median_efficiency = miner_df['efficiency'].median()
print(f"Median Efficiency = {median_efficiency}")
# label miner as 1 if its efficiency is more than median, else 0
miner_df['label'] = (miner_df['efficiency'] > median_efficiency).astype(int)
miner_df[['efficiency', 'label']].head(10)

Median Efficiency = 0.0033018258364578886


,efficiency,label
0,0.003269,0
1,0.003296,0
2,0.003253,0
3,0.003333,1
4,0.003263,0
5,0.003301,0
6,0.003360,1
7,0.003352,1
8,0.003346,1
9,0.003304,1


In [49]:
miner_df['label'].value_counts()

label
0    50
1    50
Name: count, dtype: int64

## Train/Test Split

In [50]:
feature_columns = ['blocks_mined', 'avg_transactions', 'avg_volume', 
                   'avg_fee', 'fee_volatility', 'avg_block_size', 
                   'difficulty', 'profitability', 'age']

X = miner_df[feature_columns]
y = miner_df['label']

In [51]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [52]:
print(X_train.shape)
print(X_test.shape)

(80, 9)
(20, 9)


## Scale data

In [53]:
# Scale data using normalization so that different features
# with huge numbers and small numbers have equal importance
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [54]:
# mean is close to 0 and std is 1 since we scaled it
print(X_train_scaled.mean(axis=0))
print(X_train_scaled.std(axis=0))

[-1.21263138e-12  2.63539190e-15  2.70700129e-14 -1.45439216e-15
  7.32747196e-16 -3.91908728e-15 -7.21578353e-13 -2.49411602e-14
 -4.99600361e-17]
[1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [55]:
# test data has different mean and std
# since we transformed it on the mean and std of the training data
print(X_test_scaled.mean(axis=0))
print(X_test_scaled.std(axis=0))

[-0.16666667 -0.32060631 -0.34218786 -0.50568048 -0.25456825  0.15679771
 -0.39738248 -0.50568058  0.43192155]
[0.72648316 0.91358811 1.34465054 1.27408552 1.37999707 0.87244633
 1.01367589 1.27408559 0.98881055]


## Train Model

In [56]:
from sklearn.neural_network import MLPClassifier

model = MLPClassifier(hidden_layer_sizes=(64, 32, 16, 8), activation='relu', solver='adam', alpha=0.001, random_state=42)

print(model)

MLPClassifier(alpha=0.001, hidden_layer_sizes=(64, 32, 16, 8), random_state=42)


In [57]:
# Fit the model (Training)
model.fit(X_train_scaled, y_train)

,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(64, ...)"
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.001
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",200
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True


In [58]:
# Test the accuracy of the model
# Ethan's accuracy:
# Test Accuracy = 95%
# 5-fold Cross Validation Accuracy = 48.75%
y_prediction = model.predict(X_test_scaled)
print(y_prediction)

[1 1 1 1 0 0 0 1 0 0 0 1 0 0 0 1 1 0 1 1]


In [59]:
from sklearn.metrics import accuracy_score

test_accuracy = accuracy_score(y_test, y_prediction)
print(test_accuracy)

1.0


In [60]:
from sklearn.model_selection import cross_val_score


cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5)
print(cv_scores)
print(cv_scores.mean())

[0.875  0.9375 0.8125 1.     1.    ]
0.925


In [61]:
# High cv score found? should be closer to Ethan's Accuracy
miner_df[['avg_transactions', 'avg_volume']].corrwith(miner_df['label']) 

avg_transactions    0.654373
avg_volume         -0.534180
dtype: float64

### Try to Find out the reason behind the high Cross-Validation Scores

In [62]:
# Remove the avg_transactions and avg_volume to test if they are the causes
feature_columns_reduced = ['blocks_mined',
                           'avg_fee', 'fee_volatility', 'avg_block_size',
                           'difficulty', 'profitability', 'age']

X = miner_df[feature_columns_reduced]
y = miner_df['label']

In [63]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [64]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training Data: {X_train_scaled.mean(axis=0)}")
print(f"Training Data: {X_train_scaled.std(axis=0)}")
print()
print(f"Training Data: {X_test_scaled.mean(axis=0)}")
print(f"Training Data: {X_test_scaled.std(axis=0)}")

Training Data: [-1.21263138e-12 -1.45439216e-15  7.32747196e-16 -3.91908728e-15
 -7.21578353e-13 -2.49411602e-14 -4.99600361e-17]
Training Data: [1. 1. 1. 1. 1. 1. 1.]

Training Data: [-0.16666667 -0.50568048 -0.25456825  0.15679771 -0.39738248 -0.50568058
  0.43192155]
Training Data: [0.72648316 1.27408552 1.37999707 0.87244633 1.01367589 1.27408559
 0.98881055]


In [65]:
model = MLPClassifier(hidden_layer_sizes=(64, 32, 16, 8), activation='relu', solver='adam', alpha=0.001, random_state=42)

print(model)

MLPClassifier(alpha=0.001, hidden_layer_sizes=(64, 32, 16, 8), random_state=42)


In [66]:
# Fit the model (Training)
model.fit(X_train_scaled, y_train)

d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(64, ...)"
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.001
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",200
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True


In [67]:
y_prediction = model.predict(X_test_scaled)
print(y_prediction)

[1 1 1 1 0 0 0 1 0 0 0 0 0 0 0 1 1 1 0 0]


In [68]:
test_accuracy = accuracy_score(y_test, y_prediction)
print(test_accuracy)

0.8


In [69]:
# New Cross Validation Scores after removing avg_transactions and avg_volume
cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5)
print(cv_scores)
print(cv_scores.mean())

d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


[0.625  0.5625 0.5    0.6875 0.3125]
0.5375


d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [70]:
# Rerun this with 4 early-stopping parameters added

model = MLPClassifier(
    hidden_layer_sizes=(64, 32, 16, 8),
    activation='relu',
    solver='adam',
    alpha=0.001,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    max_iter=100,
    batch_size=16
)

model.fit(X_train_scaled, y_train)
print("Epochs actually run:", model.n_iter_)

y_prediction = model.predict(X_test_scaled)
test_accuracy = accuracy_score(y_test, y_prediction)
print("Test accuracy:", test_accuracy)

cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5)
print("CV fold scores:", cv_scores)
print("CV mean:", cv_scores.mean())

Epochs actually run: 33
Test accuracy: 0.45
CV fold scores: [0.5    0.5625 0.625  0.625  0.5   ]
CV mean: 0.5625


In [71]:
def random_seed_testing(seed):
    model = MLPClassifier(
        hidden_layer_sizes=(64, 32, 16, 8),
        activation='relu',
        solver='adam',
        alpha=0.001,
        random_state=seed,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        max_iter=100,
        batch_size=16
    )

    model.fit(X_train_scaled, y_train)
    # print("Epochs actually run:", model.n_iter_)

    y_prediction = model.predict(X_test_scaled)
    test_accuracy = accuracy_score(y_test, y_prediction)
    # print("Test accuracy:", test_accuracy)

    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5)
    # print("CV fold scores:", cv_scores)
    # print("CV mean:", cv_scores.mean())
    
    print(f"Seed {seed}: test acc = {test_accuracy}, cv accuracy = {cv_scores}, epochs = {model.n_iter_}")
    
seeds = [1, 23, 35, 42, 46, 76, 81, 100]

for seed in seeds:
    random_seed_testing(seed)

Seed 1: test acc = 0.5, cv accuracy = [0.5 0.5 0.5 0.5 0.5], epochs = 39
Seed 23: test acc = 0.55, cv accuracy = [0.5  0.5  0.5  0.5  0.25], epochs = 42
Seed 35: test acc = 0.6, cv accuracy = [0.5    0.5625 0.5    0.5    0.25  ], epochs = 24
Seed 42: test acc = 0.45, cv accuracy = [0.5    0.5625 0.625  0.625  0.5   ], epochs = 33
Seed 46: test acc = 0.5, cv accuracy = [0.5 0.5 0.5 0.5 0.5], epochs = 30
Seed 76: test acc = 0.65, cv accuracy = [0.625  0.5625 0.5    0.4375 0.4375], epochs = 22
Seed 81: test acc = 0.45, cv accuracy = [0.4375 0.4375 0.5    0.5625 0.25  ], epochs = 31
Seed 100: test acc = 0.5, cv accuracy = [0.5    0.5625 0.5    0.5    0.375 ], epochs = 22


In [ ]:
# accuracy for non cv and cv seem too have dropped to around the same percentage of ~45%-65%
# accuracy for cv about matches Original result, but Seed 42 Test Accuracy dropped from 80% to 0.45%

In [73]:
# use original miner dataframe with complete feature columns and calculate the coefficient of variation
Coeffient_of_Variation = miner_df[feature_columns].std() / miner_df[feature_columns].mean()
Coeffient_of_Variation

blocks_mined        0.000035
avg_transactions    0.006210
avg_volume          0.005813
avg_fee             0.008481
fee_volatility      0.008351
avg_block_size      0.006243
difficulty          0.000062
profitability       0.008481
age                 0.586091
dtype: float64